<a href="https://colab.research.google.com/github/Jai-Kulkarni1905/Applied_Search_Intelligence/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jai-Kulkarni1905/Applied_Search_Intelligence/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Prioritize the pages that have **meaningful search visibility** but are receiving **fewer clicks than expected for their average search position**.

The baseline will rank pages using two observable signals:

* **CTR relative to position** — identifies pages that appear to under-capture clicks compared with other pages at similar positions.
* **Impressions** — ensures that the opportunity is based on meaningful search visibility rather than a small amount of noisy data.

The output is a **review-priority ranking**, not a prediction that changing the page will improve its performance.

### Reason codes
* `high_ctr_visible_page` — Optimal page with good ctr and good visibility
* `poor_position` — the page has poor position when it comes to search and thus poor visibility
* `low_ctr_visible_page` — the page has meaningful search visibility but its CTR is weak relative to pages at a similar search position.
* `insufficient_visibility` — the page does not have enough impressions for a reliable CTR-based review.
* `position_unavailable` — average position is unavailable or zero, so a position-adjusted CTR comparison cannot be made.
* `not_prioritized` — the page does not meet the conditions for the baseline review queue.

### Signal check 1 — CTR relative to position

**Signal:** CTR compared within average-position groups.

CTR should not be judged against one global threshold because pages ranking near position 1 naturally receive different click rates from pages ranking near position 10. We thus compare CTR within position tiers and check whether lower relative CTR is associated with the observed outcome.=

**Verdict:** based on the observed bucket table produced in the audit.

### Signal check 2 — Search visibility

**Signal:** GSC impressions.

Impressions provide the visibility context for the CTR signal. A low CTR based on very few impressions may simply be noise, so the rule should give greater priority to pages with meaningful existing visibility.


In [47]:
# --- Setup: connect DuckDB to the Hugging Face warehouse ---

import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

# Read the HF token from Colab Secrets
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.sql("INSTALL httpfs; LOAD httpfs;")

# Register the token as a DuckDB secret
con.sql(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
)
""")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

FACT_TABLE_GLOB = (
    f"{WAREHOUSE}/fact_content_daily_performance/**/*.parquet"
)

print("DuckDB connected and Hugging Face secret registered.")

DuckDB connected and Hugging Face secret registered.


In [48]:
march_path = (
    f"{WAREHOUSE}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet"
)

print(march_path)

hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Two small signal checks first — CTR vs position, and visibility check — both computed from the decision window only, since these are the inputs the rule actually uses. Then the score, the rank, and the CSV write.

In [49]:

df = con.sql(f"""
    SELECT *
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
""").df()

df["report_date"] = pd.to_datetime(df["report_date"])

print(f"Total rows loaded: {len(df):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows loaded: 3,611,061


2 time windows

In [50]:
feature_df = df[df["report_date"].dt.day <= 15].copy()

outcome_df = df[df["report_date"].dt.day >= 16].copy()

print("Information-window rows:", len(feature_df))
print("Outcome-window rows:", len(outcome_df))

Information-window rows: 1640237
Outcome-window rows: 1970824


page level feature table

In [51]:
features = (
    feature_df
    .groupby(["client_hash_id", "content_hash_id"])
    .agg(
        impressions=("gsc_impressions", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        clicks=("gsc_clicks", "sum")
    )
    .reset_index()
)

features["ctr"] = features["clicks"] / features["impressions"].replace(0, np.nan)
features

,client_hash_id,content_hash_id,impressions,avg_position,clicks,ctr
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,119,12.639599,1,0.008403
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,9.225529,0,0.000000
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,72,8.094074,0,0.000000
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,283,12.587226,0,0.000000
4,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,9,11.500000,0,0.000000
...,...,...,...,...,...,...
151976,client_ff644d8251367cbb,content_ffcb22817bdb0165,303,3.763376,0,0.000000
151977,client_ff644d8251367cbb,content_ffdaee2300ce7826,116,7.931378,0,0.000000
151978,client_ff644d8251367cbb,content_ffdeaa27dc03246b,814,16.119873,5,0.006143
151979,client_ff644d8251367cbb,content_ffdfde2be7d4f7d9,176,33.576000,0,0.000000


signal check 1

In [52]:

# --- CTR vs position (decision window only) ---
features = features[features["impressions"] >0]
position_bins = [0, 5, 10, 20, np.inf]
position_labels = ["1-5", "6-10", "11-20", "21+"]
features["position_bucket"] = pd.cut(features["avg_position"], bins=position_bins, labels=position_labels)

ctr_by_position = (
    features.groupby("position_bucket", observed=True)
    .agg(n=("ctr", "size"), mean_ctr=("ctr", "mean"), median_ctr=("ctr", "median"))
    .reset_index()
)
print("CTR by position bucket (n printed per bucket):\n", ctr_by_position)

CTR by position bucket (n printed per bucket):
   position_bucket      n  mean_ctr  median_ctr
0             1-5  39626  0.007392         0.0
1            6-10  46626  0.003952         0.0
2           11-20  26912  0.003337         0.0
3             21+  37511  0.002096         0.0


In [53]:
expected_ctr = (
    features
    .groupby("position_bucket", observed=False)["ctr"]
    .mean()
    .rename("expected_ctr")
)

features = features.merge(
    expected_ctr,
    on="position_bucket",
    how="left"
)

features["ctr_gap"] = (
    features["expected_ctr"] - features["ctr"]
)

features["relative_ctr"] = (
    features["ctr"] / features["expected_ctr"]
)

In [54]:
features.head()

,client_hash_id,content_hash_id,impressions,avg_position,clicks,ctr,position_bucket,expected_ctr,ctr_gap,relative_ctr
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,119,12.639599,1,0.008403,11-20,0.003337,-0.005066,2.518156
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,9.225529,0,0.000000,6-10,0.003952,0.003952,0.000000
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,72,8.094074,0,0.000000,6-10,0.003952,0.003952,0.000000
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,283,12.587226,0,0.000000,11-20,0.003337,0.003337,0.000000
4,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,9,11.500000,0,0.000000,11-20,0.003337,0.003337,0.000000


signal check 2

In [55]:
visibility_check = (
    features[features["impressions"] > 300]
    .assign(
        visibility_bucket=lambda x: pd.cut(
            x["impressions"],
            bins=[0, 500, 1000, 2500, 5000, np.inf],
            labels=[
                "1-500",
                "501-1000",
                "1,001-2,500",
                "2,501-5,000",
                "5,000+"
            ]
        )
    )
    .groupby("visibility_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions", "median")
    )
    .reset_index()
)

display(visibility_check)

,visibility_bucket,n,median_impressions
0,1-500,11136,387.0
1,501-1000,14799,713.0
2,"1,001-2,500",14699,1517.0
3,"2,501-5,000",6845,3371.0
4,"5,000+",5423,7975.0


building obbserved outcome

In [56]:
VISIBILITY_THRESHOLD = 500

# Start from the feature table already created above
baseline_df = features.copy()

# Position availability
baseline_df["position_available"] = (
    baseline_df["avg_position"].notna()
    & (baseline_df["avg_position"] > 0)
)

# Meaningful visibility
baseline_df["high_visibility"] = (
    baseline_df["impressions"] >= VISIBILITY_THRESHOLD
)

# Use the observed mean CTR for each position bucket
# CTR relative to the position-group expectation

In [57]:
baseline_df.head()

,client_hash_id,content_hash_id,impressions,avg_position,clicks,ctr,position_bucket,expected_ctr,ctr_gap,relative_ctr,position_available,high_visibility
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,119,12.639599,1,0.008403,11-20,0.003337,-0.005066,2.518156,True,False
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,9.225529,0,0.000000,6-10,0.003952,0.003952,0.000000,True,False
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,72,8.094074,0,0.000000,6-10,0.003952,0.003952,0.000000,True,False
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,283,12.587226,0,0.000000,11-20,0.003337,0.003337,0.000000,True,False
4,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,9,11.500000,0,0.000000,11-20,0.003337,0.003337,0.000000,True,False


In [58]:
# Reason codes
baseline_df["reason_code"] = np.select(
    [
        ~baseline_df["position_available"],

        baseline_df["position_available"]
        & ~baseline_df["high_visibility"]
        & (baseline_df['position_bucket'].isin(["11-20", "21+"])),

        baseline_df["position_available"]
        & ~baseline_df["high_visibility"]
        & (baseline_df['position_bucket'].isin(["1-5", "6-10"])),

        baseline_df["position_available"]
        & baseline_df["high_visibility"]
        & (baseline_df["ctr"] < baseline_df['expected_ctr']),

        baseline_df["position_available"]
        & baseline_df["high_visibility"]
        & (baseline_df["ctr"] >= baseline_df['expected_ctr']),

        baseline_df["position_available"]
        & ~baseline_df["high_visibility"]
        & (baseline_df["ctr"] >= baseline_df['expected_ctr'])
        & (baseline_df['position_bucket'].isin(["11-20", "21+"])),
    ],
    [
        "position_unavailable",
        "poor_position",
        "insufficient_visibility",
        "low_ctr_visible_page",
        "high_ctr_visible_page",
        "high_ctr_poor_position"
    ],
    default="not_prioritized"
)


In [59]:
# Review importance
baseline_df["review_priority"] = np.select(
    [
        baseline_df["reason_code"] == "low_ctr_visible_page",
        baseline_df["reason_code"] == "insufficient_visibility",
        baseline_df["reason_code"] == "high_ctr_visible_page",
        baseline_df["reason_code"] == "position_unavailable",
        baseline_df["reason_code"] == "poor_position",
    ],
    ["med","med", "na", "high","high"],default="na"
)


In [60]:
baseline_df.reason_code.value_counts()

,count
reason_code,
insufficient_visibility,58860
poor_position,49999
low_ctr_visible_page,32080
high_ctr_visible_page,9736
position_unavailable,1306


In [61]:
# Display the resulting classification
print("Reason-code counts:")
display(
    baseline_df["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="n")
)

print("\nReview-priority counts:")
display(
    baseline_df["review_priority"]
    .value_counts()
    .rename_axis("review_priority")
    .reset_index(name="n")
)

display(
    baseline_df[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "ctr",
            "avg_position",
            "position_bucket",
            "expected_ctr",
            "relative_ctr",
            "reason_code",
            "review_priority"
        ]
    ].head(20)
)

Reason-code counts:


,reason_code,n
0,insufficient_visibility,58860
1,poor_position,49999
2,low_ctr_visible_page,32080
3,high_ctr_visible_page,9736
4,position_unavailable,1306



Review-priority counts:


,review_priority,n
0,med,90940
1,high,51305
2,na,9736


,client_hash_id,content_hash_id,impressions,ctr,avg_position,position_bucket,expected_ctr,relative_ctr,reason_code,review_priority
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,119,0.008403,12.639599,11-20,0.003337,2.518156,poor_position,high
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,0.000000,9.225529,6-10,0.003952,0.000000,insufficient_visibility,med
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,72,0.000000,8.094074,6-10,0.003952,0.000000,insufficient_visibility,med
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,283,0.000000,12.587226,11-20,0.003337,0.000000,poor_position,high
4,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,9,0.000000,11.500000,11-20,0.003337,0.000000,poor_position,high
5,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,66,0.000000,12.282875,11-20,0.003337,0.000000,poor_position,high
6,client_0797ff3a1fc9a6a5,content_1fea2f270f3c1350,2,0.000000,3.500000,1-5,0.007392,0.000000,insufficient_visibility,med
7,client_0797ff3a1fc9a6a5,content_27f8100281413b37,11,0.000000,8.083333,6-10,0.003952,0.000000,insufficient_visibility,med
8,client_0797ff3a1fc9a6a5,content_2959111291cf661f,2,0.000000,28.500000,21+,0.002096,0.000000,poor_position,high
9,client_0797ff3a1fc9a6a5,content_2f719399052f18fc,8,0.000000,21.250000,21+,0.002096,0.000000,poor_position,high


In [62]:
baseline_df.head()

,client_hash_id,content_hash_id,impressions,avg_position,clicks,ctr,position_bucket,expected_ctr,ctr_gap,relative_ctr,position_available,high_visibility,reason_code,review_priority
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,119,12.639599,1,0.008403,11-20,0.003337,-0.005066,2.518156,True,False,poor_position,high
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,9.225529,0,0.000000,6-10,0.003952,0.003952,0.000000,True,False,insufficient_visibility,med
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,72,8.094074,0,0.000000,6-10,0.003952,0.003952,0.000000,True,False,insufficient_visibility,med
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,283,12.587226,0,0.000000,11-20,0.003337,0.003337,0.000000,True,False,poor_position,high
4,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,9,11.500000,0,0.000000,11-20,0.003337,0.003337,0.000000,True,False,poor_position,high


In [63]:
# FINAL BASELINE ACTION SCORE
action_df = baseline_df.copy()

# 1. CTR NEED
# Relative shortfall from the MEAN CTR of the page's position bucket
# 0     = CTR >= expected CTR
# > 0   = CTR below expected CTR
# Missing position cannot be evaluated on CTR

action_df["ctr_need"] = np.where(
    action_df["position_available"]
    & action_df["expected_ctr"].notna()
    & (action_df["expected_ctr"] > 0),

    (
        (action_df["expected_ctr"] - action_df["ctr"])
        / action_df["expected_ctr"]
    ).clip(lower=0),

    0.0
)

# 2. VISIBILITY SIGNAL
# Use log1p so the scale is finite
# Higher visibility = stronger evidence that the page has meaningful search exposure
# Zero impressions remains explicitly identifiable and will receive a finite priority contribution below

log_impressions = np.log1p(
    action_df["impressions"].fillna(0)
)

max_log_impressions = log_impressions.max()

if max_log_impressions > 0:
    action_df["visibility_strength"] = (
        log_impressions / max_log_impressions
    )
else:
    action_df["visibility_strength"] = 0.0


# 3. POSITION NEED
# Poorer position = greater need
# Missing position receives the maximum finite position need

valid_positions = action_df.loc[
    action_df["position_available"],
    "avg_position"
]

if len(valid_positions) > 0:

    max_log_position = np.log1p(
        valid_positions.max()
    )

    action_df["position_need"] = np.where(
        action_df["position_available"],
        np.log1p(
            action_df["avg_position"].clip(lower=0)
        ) / max_log_position,
        1.0
    )

else:
    action_df["position_need"] = 1.0


# 4. ZERO-VISIBILITY NEED
# Zero-visibility pages still need attention

action_df["zero_visibility_need"] = np.where(
    action_df["impressions"].fillna(0) == 0,
    1.0,
    0.0
)

In [64]:

# 5. FINAL EQUAL-WEIGHT SCORE
# Four equally weighted observable components:
#   CTR need
#   visibility strength
#   position need
#   zero-visibility need


action_df["action_score"] = (
    action_df["ctr_need"]
    + action_df["visibility_strength"]
    + action_df["position_need"]
    + action_df["zero_visibility_need"]
)


# 6. ACTION LABEL

q_high = action_df["action_score"].quantile(0.75)
q_medium = action_df["action_score"].quantile(0.50)

action_df["action"] = np.select(
    [
        action_df["action_score"] >= q_high,
        action_df["action_score"] >= q_medium
    ],
    [
        "high",
        "medium"
    ],
    default="low"
)


# 7. RANK

action_df = action_df.sort_values(
    ["action_score", "impressions", "content_hash_id"],
    ascending=[False, False, True]
).reset_index(drop=True)

action_df["rank"] = np.arange(
    1,
    len(action_df) + 1
)


In [65]:

# 8. FINAL RANKED QUEUE

action_output = action_df[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action_score",
        "action",
        "reason_code",
        "impressions",
        "ctr",
        "avg_position",
        "expected_ctr",
        "relative_ctr",
        "ctr_need",
        "visibility_strength",
        "position_need",
        "zero_visibility_need"
    ]
].copy()


print("Action score summary:")
display(
    action_output["action_score"].describe()
)

print("\nAction distribution:")
display(
    action_output["action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="n")
)

print("\nTop 20:")
display(
    action_output.head(20)
)

Action score summary:


,action_score
count,151981.000000
mean,1.558558
std,0.387269
min,0.161058
25%,1.354114
50%,1.635236
75%,1.848968
max,2.543081



Action distribution:


,action,n
0,low,75990
1,high,37996
2,medium,37995



Top 20:


,rank,client_hash_id,content_hash_id,action_score,action,reason_code,impressions,ctr,avg_position,expected_ctr,relative_ctr,ctr_need,visibility_strength,position_need,zero_visibility_need
0,1,client_23a62021009f63c4,content_559cdd76da9306de,2.543081,high,low_ctr_visible_page,62418,0.000032,37.866104,0.002096,0.015287,0.984713,0.920693,0.637675,0.0
1,2,client_23a62021009f63c4,content_c367b0ca57f3559b,2.540167,high,low_ctr_visible_page,33909,0.000029,49.827374,0.002096,0.014070,0.985930,0.869816,0.684421,0.0
2,3,client_23a62021009f63c4,content_295e883e0e86ca3c,2.533368,high,low_ctr_visible_page,16488,0.000000,62.671594,0.002096,0.000000,1.000000,0.809695,0.723674,0.0
3,4,client_23a62021009f63c4,content_96e6613b42b52c42,2.532423,high,low_ctr_visible_page,34111,0.000029,47.456360,0.002096,0.013986,0.986014,0.870311,0.676098,0.0
4,5,client_23a62021009f63c4,content_6aa54d6bbdbf6f24,2.530907,high,low_ctr_visible_page,27832,0.000072,58.491316,0.002096,0.034284,0.965716,0.853348,0.711843,0.0
5,6,client_23a62021009f63c4,content_1162dc8495e06dfb,2.506777,high,low_ctr_visible_page,33754,0.000030,41.070564,0.002096,0.014134,0.985866,0.869434,0.651478,0.0
6,7,client_23a62021009f63c4,content_bbf8e4d669f253cf,2.502474,high,low_ctr_visible_page,18571,0.000000,49.374079,0.002096,0.000000,1.000000,0.819614,0.682860,0.0
7,8,client_23a62021009f63c4,content_8239af50cfd75d26,2.501088,high,low_ctr_visible_page,33316,0.000030,40.018068,0.002096,0.014320,0.985680,0.868345,0.647064,0.0
8,9,client_23a62021009f63c4,content_ab91e088440ace78,2.496848,high,low_ctr_visible_page,38414,0.000104,44.809035,0.002096,0.049679,0.950321,0.880217,0.666310,0.0
9,10,client_23a62021009f63c4,content_67a19b4e8f52924e,2.485658,high,low_ctr_visible_page,19325,0.000000,43.876153,0.002096,0.000000,1.000000,0.822932,0.662725,0.0


In [66]:
from google.colab import drive

## saving to drive because facing issues with saving files to git repo, might see later to solve

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Save directly to your Drive root
action_output.to_csv('/content/drive/MyDrive/baseline_action_score.csv', index=False)
print("Saved to Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Google Drive!


## precision @ k

In [67]:

# Construct the held-out label( DAY 16+ ONLY)

heldout = (
    outcome_df
    .groupby(["client_hash_id", "content_hash_id"])
    .agg(
        outcome_impressions=("gsc_impressions", "sum")
    )
    .reset_index()
)

# First-half impressions are already in `features`
evaluation_df = (
    action_df[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "action_score",
            "action",
            "reason_code",
            "impressions"
        ]
    ]
    .merge(
        features[
            [
                "client_hash_id",
                "content_hash_id",
                "impressions"
            ]
        ].rename(
            columns={"impressions": "information_impressions"}
        ),
        on=["client_hash_id", "content_hash_id"],
        how="left"
    )
    .merge(
        heldout,
        on=["client_hash_id", "content_hash_id"],
        how="left"
    )
)


#  Observed held-out outcome
# A page is labelled as declining when its second-half impressions are lower than its first-half impressions.

evaluation_df["is_declining"] = (
    evaluation_df["outcome_impressions"]
    < evaluation_df["information_impressions"]
)

# Only evaluate pages for which the held-out outcome exists.
evaluation_df = evaluation_df[
    evaluation_df["outcome_impressions"].notna()
].copy()


# Precision@K

def precision_at_k(df, k):
    top_k = df.sort_values(
        "rank",
        ascending=True
    ).head(k)

    if len(top_k) == 0:
        return np.nan

    return top_k["is_declining"].mean()


K_VALUES = [10, 25, 50, 100, 250, 500]

precision_results = []

for k in K_VALUES:
    precision_results.append(
        {
            "K": k,
            "n_evaluated": min(k, len(evaluation_df)),
            "precision_at_k": precision_at_k(
                evaluation_df,
                k
            )
        }
    )

precision_at_k_df = pd.DataFrame(
    precision_results
)

# Held-out base rate for context
base_rate = evaluation_df["is_declining"].mean()

print("\nHeld-out evaluation:")
print(f"Pages with observed outcome: {len(evaluation_df):,}")
print(f"Held-out decline rate: {base_rate:.3f}")

display(precision_at_k_df)


Held-out evaluation:
Pages with observed outcome: 141,467
Held-out decline rate: 0.396


,K,n_evaluated,precision_at_k
0,10,10,0.900
1,25,25,0.840
2,50,50,0.800
3,100,100,0.780
4,250,250,0.768
5,500,500,0.750


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [68]:
# TOP-20 REVIEW

top20 = action_df.sort_values("rank").head(20)

for _, row in top20.iterrows():

    # Action label
    action_map = {
        "high": "review_high_priority",
        "medium": "review_medium_priority",
        "low": "review_low_priority"
    }

    action = action_map.get(
        row["action"],
        "no_action"
    )

    reason = row["reason_code"]

    # Why
    if reason == "low_ctr_visible_page":
        why = (
            f"Visible with {row['impressions']:,.0f} impressions; "
            f"CTR ({row['ctr']:.3f}) is below the position-group mean "
            f"({row['expected_ctr']:.3f})."
        )

    elif reason == "high_ctr_visible_page":
        why = (
            f"Has {row['impressions']:,.0f} impressions and CTR "
            f"({row['ctr']:.3f}) meets or exceeds the position-group "
            f"mean ({row['expected_ctr']:.3f})."
        )

    elif reason == "insufficient_visibility":
        why = (
            f"CTR is available, but visibility is limited at "
            f"{row['impressions']:,.0f} impressions."
        )

    elif reason == "position_unavailable":
        why = (
            f"CTR cannot be compared by position because average "
            f"position is unavailable or zero; visibility is "
            f"{row['impressions']:,.0f} impressions."
        )

    elif reason == "poor_position":
        why = (
            f"Has limited visibility ({row['impressions']:,.0f} impressions) "
            f"and ranks in position bucket {row['position_bucket']}."
        )

    elif reason == "high_ctr_poor_position":
        why = (
            f"CTR ({row['ctr']:.3f}) is at or above its position-group "
            f"mean, but the page ranks in position bucket {row['position_bucket']}."
        )

    else:
        why = (
            f"Score reflects its observed visibility, CTR and position "
            f"signals."
        )

    # --------------------------------------------------------
    # Confidence
    # --------------------------------------------------------
    if reason == "low_ctr_visible_page":
        confidence = (
            "high — visibility and below-position CTR provide "
            "two supporting signals"
        )

    elif reason == "position_unavailable":
        confidence = (
            "low — position-adjusted CTR evidence is unavailable"
        )

    elif reason == "insufficient_visibility":
        confidence = (
            "low — limited impressions reduce the strength of the evidence"
        )

    elif reason == "high_ctr_visible_page":
        confidence = (
            "medium — strong CTR evidence, but no CTR shortfall"
        )

    elif reason == "high_ctr_poor_position":
        confidence = (
            "medium — poor position creates review need despite adequate CTR"
        )

    elif reason == "poor_position":
        confidence = (
            "medium — limited visibility and weak position support review"
        )

    else:
        confidence = (
            "low — limited evidence from the available signals"
        )

    # --------------------------------------------------------
    # What could make it wrong
    # --------------------------------------------------------
    if reason == "low_ctr_visible_page":
        wrong = (
            "the CTR gap reflects search-intent mismatch or SERP conditions "
            "rather than a fixable page issue"
        )

    elif reason == "high_ctr_visible_page":
        wrong = (
            "the apparent strong CTR does not reflect the page's broader "
            "search performance"
        )

    elif reason == "insufficient_visibility":
        wrong = (
            "low impressions are temporary or the page has intentionally "
            "limited search demand"
        )

    elif reason == "position_unavailable":
        wrong = (
            "the missing position reflects data limitations rather than "
            "a genuine positioning problem"
        )

    elif reason == "high_ctr_poor_position":
        wrong = (
            "low visibility is caused by limited search demand rather than "
            "a page-level ranking opportunity"
        )

    elif reason == "poor_position":
        wrong = (
            "the poor position reflects low-demand queries rather than "
            "a page-level opportunity"
        )

    else:
        wrong = (
            "the observed signals do not represent a meaningful page-level "
            "opportunity"
        )

    # --------------------------------------------------------
    # Print
    # --------------------------------------------------------
    print(
        f"Rank {int(row['rank'])} | "
        f"content_hash_id={row['content_hash_id']} | "
        f"action={action} | "
        f"reason={reason} | "
        f"score={row['action_score']:.3f}"
    )

    print(f"  Why: {why}")
    print(f"  Confidence: {confidence}")
    print(f"  Would be wrong if: {wrong}")

Rank 1 | content_hash_id=content_559cdd76da9306de | action=review_high_priority | reason=low_ctr_visible_page | score=2.543
  Why: Visible with 62,418 impressions; CTR (0.000) is below the position-group mean (0.002).
  Confidence: high — visibility and below-position CTR provide two supporting signals
  Would be wrong if: the CTR gap reflects search-intent mismatch or SERP conditions rather than a fixable page issue
Rank 2 | content_hash_id=content_c367b0ca57f3559b | action=review_high_priority | reason=low_ctr_visible_page | score=2.540
  Why: Visible with 33,909 impressions; CTR (0.000) is below the position-group mean (0.002).
  Confidence: high — visibility and below-position CTR provide two supporting signals
  Would be wrong if: the CTR gap reflects search-intent mismatch or SERP conditions rather than a fixable page issue
Rank 3 | content_hash_id=content_295e883e0e86ca3c | action=review_high_priority | reason=low_ctr_visible_page | score=2.533
  Why: Visible with 16,488 impress

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [69]:
# LEAKAGE CHECK

print("=== INFORMATION / FUTURE-WINDOW CHECK ===")

# 1. Confirm feature window
feature_dates = pd.to_datetime(feature_df["report_date"])

print(
    f"Feature window: {feature_dates.min().date()} "
    f"to {feature_dates.max().date()}"
)

assert feature_dates.dt.day.max() <= 15, \
    "LEAKAGE: feature_df contains day 16+ data"

print("PASS: all scoring features come from days 1–15.")


# 2. Confirm outcome window is separate
outcome_dates = pd.to_datetime(outcome_df["report_date"])

print(
    f"Outcome window: {outcome_dates.min().date()} "
    f"to {outcome_dates.max().date()}"
)

assert outcome_dates.dt.day.min() >= 16, \
    "LEAKAGE: outcome_df contains day 1–15 data"

print("PASS: held-out outcome uses days 16+ only.")


# SCORING DATAFRAME COLUMN CHECK

print("\n=== SCORING INPUT CHECK ===")

# Columns actually used to construct the score
score_inputs = {
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "expected_ctr",
    "relative_ctr"
}

print("Score-related columns:")
print(sorted(score_inputs))


# Future/outcome columns that must NOT influence scoring
future_columns = {
    "outcome_impressions",
    "is_declining",
    "trend_pct",
    "trend_direction"
}

leaked_future_columns = (
    future_columns
    .intersection(action_df.columns)
)

print("\nFuture/outcome columns present in action_df:")
print(sorted(leaked_future_columns))

assert not leaked_future_columns, \
    f"LEAKAGE: future/outcome columns found in action_df: {leaked_future_columns}"

print("PASS: no held-out outcome fields are present in scoring dataframe.")


# PRODUCT FLAG CHECK

print("\n=== PRODUCT FLAG CHECK ===")

# Search for columns whose names indicate product flags.
product_flag_keywords = [
    "flag",
    "refresh",
    "quick_win",
    "ctr_fix",
    "product"
]

product_flag_columns = [
    col for col in action_df.columns
    if any(
        keyword in col.lower()
        for keyword in product_flag_keywords
    )
]

print("Potential product/flag columns found:")
print(product_flag_columns)

# These should not be inputs to the score.
score_feature_columns = {
    "ctr_need",
    "visibility_strength",
    "position_need",
    "zero_visibility_need",
    "action_score"
}

used_flag_columns = [
    col for col in product_flag_columns
    if col in score_feature_columns
]

assert not used_flag_columns, \
    f"LEAKAGE: product flag used in score: {used_flag_columns}"

print("PASS: no product flags are used by the action score.")


# FINAL CHECK

print("\n=== FINAL LEAKAGE STATUS ===")
print("PASS: baseline ranking uses first-half observed features only.")
print("PASS: held-out day 16+ data is used only for evaluation.")
print("PASS: product flags are not used in scoring.")

=== INFORMATION / FUTURE-WINDOW CHECK ===
Feature window: 2026-03-01 to 2026-03-15
PASS: all scoring features come from days 1–15.
Outcome window: 2026-03-16 to 2026-03-31
PASS: held-out outcome uses days 16+ only.

=== SCORING INPUT CHECK ===
Score-related columns:
['avg_position', 'clicks', 'ctr', 'expected_ctr', 'impressions', 'relative_ctr']

Future/outcome columns present in action_df:
[]
PASS: no held-out outcome fields are present in scoring dataframe.

=== PRODUCT FLAG CHECK ===
Potential product/flag columns found:
[]
PASS: no product flags are used by the action score.

=== FINAL LEAKAGE STATUS ===
PASS: baseline ranking uses first-half observed features only.
PASS: held-out day 16+ data is used only for evaluation.
PASS: product flags are not used in scoring.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.